In [ ]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import pickle

%run Model_functions.ipynb

In [ ]:
experiment_registry = {}
with open('sentinel_test.pkl', 'rb') as f:
    experiment_registry = pickle.load(f)

# JUST BASIC TESTING

In [ ]:
# Get first experiment
print_registry_entry(experiment_registry[0])

In [ ]:
query_results = query_experiments(experiment_registry, model_type="rf", is_groups=True)
print_registry_entry(query_results[0])

## Best experiments

In [ ]:
best_results = filter_best_experiments(registry_to_dict(experiment_registry))
tab_df = tabulate_results(best_results, top_n=10)

**With query filter**

In [ ]:
query_results = query_experiments(experiment_registry, model_type="rf", is_groups=True)
best_results  = filter_best_experiments(registry_to_dict(query_results))
tab_df        = tabulate_results(best_results, top_n=10)

## Various queries

In [ ]:
# Query ALL
tab_df = tabulate_registry(experiment_registry)

In [ ]:
# Query All acceptable experiments
tab_df = tabulate_registry(experiment_registry, verdict="\u2705")

In [ ]:
# Query All grouped experiments with test R² > 0.25
tab_df = tabulate_registry(experiment_registry, is_groups=True, min_test_r2=0.25)

In [ ]:
# Query Any interaction terms
tab_df = tabulate_registry(experiment_registry,features_contains_pattern="height_x_",)

In [ ]:
# Interaction terms included
tab_df = tabulate_registry(experiment_registry, features_contains=['height_x_EVI', 'height'])

In [ ]:
tab_df = tabulate_registry(experiment_registry, model_type="lasso")

In [ ]:
# Query All grouped RF no-grid
tab_df = tabulate_registry(experiment_registry, model_type="rf", is_groups=True, is_grid=False)

In [ ]:
# Query All feature combinations for a dataset+model combo
tab_df = tabulate_registry(experiment_registry, dataset="SENTINEL+TANDEMX", model_type="rf")

In [ ]:
# Query All XGBoost
tab_df = tabulate_registry(experiment_registry, model_type="xgboost")

In [ ]:
# Query All LGBM
tab_df = tabulate_registry(experiment_registry, model_type="lgbm")

In [ ]:
# Query All MERF
tab_df = tabulate_registry(experiment_registry, model_type="merf")

## VERIFY PLOTS

In [ ]:
%run Model_functions.ipynb

### MODEL TRAINING PLOTS

#### Random forest

In [ ]:
tab_df = tabulate_registry(experiment_registry, model_type="rf", is_groups=True, is_grid=False)

In [ ]:
plot_oob_curve(tab_df.iloc[0])

In [ ]:
query_results = query_experiments(experiment_registry, model_type="rf", is_groups=True, is_grid=False)
plot_oob_curve(query_results[0]["results"], label=query_results[0]["label"])

#### XGBoost

In [ ]:
tab_df = tabulate_registry(experiment_registry, model_type="xgboost", is_groups=True, is_grid=False)

In [ ]:
plot_xgboost_loss(tab_df.iloc[0])

In [ ]:
query_results = query_experiments(experiment_registry, model_type="xgboost", is_groups=True, is_grid=False)
plot_xgboost_loss(query_results[0]["results"], label=query_results[0]["label"])

#### Light GBM

In [ ]:
tab_df = tabulate_registry(experiment_registry, model_type="lgbm", is_groups=True, is_grid=False)

In [ ]:
plot_lgbm_loss(tab_df.iloc[0])

In [ ]:
query_results = query_experiments(experiment_registry, model_type="lgbm", is_groups=True, is_grid=False)
plot_lgbm_loss(query_results[0]["results"], label=query_results[0]["label"])

#### MERF

In [ ]:
tab_df = tabulate_registry(experiment_registry, model_type="merf", is_groups=True, is_grid=False)

In [ ]:
plot_merf_convergence(tab_df.iloc[0])

In [ ]:
query_results = query_experiments(experiment_registry, model_type="merf", is_groups=True, is_grid=False)
plot_merf_convergence(query_results[0]["results"], label=query_results[0]["label"])

## MODEL PREDICTION PLOTS

In [ ]:
query_results = query_experiments(experiment_registry, model_type="rf", is_groups=True)

e   = query_results[0]         # or query_results[0]
res = e["results"]
lbl = e["label"]

print(res.get('grouping'))
print(res.get('model_name'))

In [ ]:
# Predicted vs Actual
plot_predicted_vs_actual(res['y_pred'], res['y_test_values'], label=lbl)

In [ ]:
# CV scores
plot_cv_scores(res['cv_scores'],
               res.get('group_cv_scores'),
               res.get('fold_sites'),
               label=lbl)

In [ ]:
# Residuals by site
%run Model_functions.ipynb
plot_residuals_by_site(res['residuals'], res['groups_test'], label=lbl)

In [ ]:
# AGB distribution
plot_agb_distribution(res['y_pred'], res['y_test_values'], label=lbl)

In [ ]:
# Error vs AGB magnitude
plot_error_vs_agb(res['y_pred'], res['y_test_values'], res['residuals'], label=lbl)

In [ ]:
residual_analysis(res['y_pred'], res['residuals'])

In [ ]:
residual_analysis_ver2(res['y_pred'], res['residuals'])

# EMIT EXPERIMENTS

In [ ]:
emit_expr_registry = {}
with open('emit_expr_registry.pkl', 'rb') as f:
    emit_expr_registry = pickle.load(f)

In [ ]:
# EMIT bands + height
query_experiments(emit_expr_registry, has_emit=True, features_contains=['height'])

# EMIT bands + height + species
query_experiments(emit_expr_registry, has_emit=True, features_contains=['height', 'species'])

# EMIT bands + any index
query_experiments(emit_expr_registry, has_emit=True, features_contains=['EVI'])

# No EMIT, just indices and height
query_experiments(emit_expr_registry, has_emit=False, features_contains=['height', 'EVI', 'NBR'])

# Any EMIT band
query_experiments(emit_expr_registry, features_contains_pattern="EMIT_")

# Combine — height interactions + grouped
query_experiments(emit_expr_registry, 
                  features_contains_pattern="height_x_",
                  is_groups=True)

# Any EMIT band + any interaction terms
query_experiments(emit_expr_registry,
                  has_emit=True,
                  features_contains_pattern="height_x_", is_groups=True)

# Query All feature combinations for a dataset+model combo
query_results = query_experiments(emit_expr_registry, dataset="EMIT+TANDEMX", model_type="rf")
tab_df = tabulate_results([(e["label"], e["results"]) for e in query_results])

In [ ]:
# EMIT bands only — no height, no indices, no interactions
query_experiments(emit_expr_registry, emit_only=True)

# EMIT bands only + grouped RF
query_experiments(emit_expr_registry, emit_only=True, model_type="rf", is_groups=True)

## Check if the loaded values are printable

### Print the first experiment.

In [ ]:
print_registry_entry(emit_expr_registry[0])

### Display the best experiments from the loaded results

In [ ]:
best_results = filter_best_experiments(registry_to_dict(emit_expr_registry))
tab_df = tabulate_results(best_results, top_n=10)

### Display experiment from a chosen row among the best list

In [ ]:
print_experiment_from_row(tab_df.iloc[0])

# SENTINEL-2 EXPERIMENTS

In [ ]:
%run Model_functions.ipynb

In [ ]:
sent_expr_registry = {}
with open('sentinel_expr_registry.pkl', 'rb') as f:
    sent_expr_registry = pickle.load(f)

## Check if the loaded values are printable

### Print the first experiment.

In [ ]:
print_registry_entry(sent_expr_registry[0])

### Display the best experiments from the loaded results

In [ ]:
best_results = filter_best_experiments(registry_to_dict(sent_expr_registry))
tab_df = tabulate_results(best_results, top_n=10)

### Display experiment from a chosen row among the best list

In [ ]:
print_experiment_from_row(tab_df.iloc[0])

# FUNCTIONS FOR GLOBAL DICT

In [ ]:
loaded_dict = {}
with open('sentinel_expr_dictionary.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)

### Print the first experiment.

In [ ]:
%run Model_functions.ipynb
key_list = list(loaded_dict.keys())
print_experiment(loaded_dict, key_list[0])

### Display the best experiments from the loaded results

In [ ]:
%run Model_functions.ipynb
best_results = filter_best_experiments(loaded_dict)
tab_df = tabulate_results(best_results, top_n=10)

### Display experiment from a chosen row among the best list

In [ ]:
%run Model_functions.ipynb
print_experiment_from_row(tab_df.iloc[0])